In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')

Criação do banco SQLite a partir do arquivo banco_python_final.parquet

In [ ]:
import shutil
import sqlite3
from pathlib import Path

import pandas as pd
from tqdm.auto import tqdm


ENTRADA = Path(
    "/content/drive/MyDrive/TCC_Chatbot/datasets/merged/banco_python_final.parquet"
)

SQLITE_LOCAL = Path(
    "/content/banco_respostas.sqlite"
)

SQLITE_DRIVE = Path(
    "/content/drive/MyDrive/TCC_Chatbot/SQLite/banco_respostas.sqlite"
)

SQLITE_DRIVE.parent.mkdir(
    parents=True,
    exist_ok=True
)

if SQLITE_LOCAL.exists():
    SQLITE_LOCAL.unlink()

print("Carregando banco...")

df = pd.read_parquet(ENTRADA)

print(f"Total de registros: {len(df):,}")

df = df.reset_index(names="id")

conn = sqlite3.connect(SQLITE_LOCAL)

cursor = conn.cursor()

cursor.execute("PRAGMA synchronous=NORMAL")
cursor.execute("PRAGMA temp_store=MEMORY")

try:

    cursor.execute("""
    CREATE TABLE IF NOT EXISTS respostas (

        id INTEGER PRIMARY KEY,

        question TEXT NOT NULL,

        answer TEXT NOT NULL,

        code TEXT,

        source TEXT,

        UNIQUE(question, answer)

    )
    """)

    conn.commit()

    BATCH_SIZE = 5000

    total = 0

    for inicio in tqdm(
        range(0, len(df), BATCH_SIZE),
        desc="Importando registros"
    ):

        fim = min(
            inicio + BATCH_SIZE,
            len(df)
        )

        lote = df.iloc[inicio:fim]

        registros = [

            (

                row.id,
                row.question,
                row.answer,
                row.code if pd.notna(row.code) else "",
                row.source

            )

            for row in lote.itertuples(index=False)

        ]

        cursor.executemany(
            """
            INSERT INTO respostas
            (
                id,
                question,
                answer,
                code,
                source
            )
            VALUES (?, ?, ?, ?, ?)
            """,
            registros
        )

        conn.commit()

        total += len(registros)

    print("\nCriando índices...")

    cursor.execute("""
    CREATE INDEX IF NOT EXISTS idx_source
    ON respostas(source)
    """)

    cursor.execute("""
    CREATE INDEX IF NOT EXISTS idx_question
    ON respostas(question)
    """)

    conn.commit()

    cursor.execute(
        "SELECT COUNT(*) FROM respostas"
    )

    quantidade = cursor.fetchone()[0]

    print(f"\nTotal gravado no SQLite: {quantidade:,}")

    print("\nCompactando banco (VACUUM)...")

    cursor.execute("VACUUM")

    conn.commit()

finally:

    conn.close()

print("\nCopiando banco para o Google Drive...")

shutil.copy2(
    SQLITE_LOCAL,
    SQLITE_DRIVE
)

print("\nBanco SQLite criado com sucesso!")

print(f"Total importado: {total:,}")

print("Banco salvo em:")

print(SQLITE_DRIVE)

Validação do banco SQLite

In [ ]:
import sqlite3
import numpy as np

conn = sqlite3.connect(
    "/content/drive/MyDrive/TCC_Chatbot/SQLite/banco_respostas.sqlite"
)

cursor = conn.cursor()

cursor.execute("""
SELECT id, question, answer, source
FROM respostas
LIMIT 5
""")

for row in cursor.fetchall():

    print(f"ID: {row[0]}")
    print(f"Pergunta: {row[1][:80]}...")
    print(f"Resposta: {row[2][:80]}...")
    print(f"Source: {row[3]}")
    print("-" * 50)

conn.close()

Analisando os índices do arquivo banco_python_final.parquet

In [ ]:
import pandas as pd

df = pd.read_parquet(
    "/content/drive/MyDrive/TCC_Chatbot/datasets/merged/banco_python_final.parquet"
)

mudancas = df["source"] != df["source"].shift()

print(
    df.loc[mudancas, ["source"]]
    .reset_index()
)

Verificando se os índices correspondem a mesma sequência do banco_final_python.parquet

In [ ]:
import sqlite3
import pandas as pd

conn = sqlite3.connect(
    "/content/drive/MyDrive/TCC_Chatbot/SQLite/banco_respostas.sqlite"
)

df_sqlite = pd.read_sql_query(
    """
    SELECT id, source
    FROM respostas
    ORDER BY id
    """,
    conn
)

conn.close()

mudancas = df_sqlite["source"] != df_sqlite["source"].shift()

print(
    df_sqlite.loc[mudancas]
)